In [1]:
# === PhantomCrypt cover-admissibility check — self-contained, no main(), no argparse ===
import hashlib, pickle, re

P = 2**256 - 2**32 - 977  # field prime

def Hfield_word(w, p=P):
    return int.from_bytes(hashlib.sha3_256(w.encode()).digest(), "big") % p

def normalize(text):
    return re.findall(r"\S+", text)

def is_admissible(words, k, p=P):
    L = len(words)
    if L < k + 1:
        return False, "length"
    if len(set(words)) < k:
        return False, "non-degeneracy"
    shares = {Hfield_word(w, p) for w in words}
    shares.discard(0)
    if len(shares) < k:
        return False, "share-separation"
    return True, "ok"

def sweep(corpus, L, k_mode):
    total, passed = 0, 0
    fail_reasons = {"length": 0, "non-degeneracy": 0, "share-separation": 0}
    for _, text in corpus.items():
        w = normalize(text)
        for start in range(0, len(w) - L + 1, L):
            window = w[start:start + L]
            k = 5 if k_mode == "const" else max(2, (L + 7) // 8)
            ok, reason = is_admissible(window, k)
            total += 1
            if ok: passed += 1
            else:  fail_reasons[reason] += 1
    return total, passed, fail_reasons

with open("corpus.pkl", "rb") as f:
    corpus = pickle.load(f)
total_tokens = sum(len(normalize(t)) for t in corpus.values())
print(f"Corpus: {len(corpus)} articles, {total_tokens} total tokens\n")
print("Admissibility pass rate on UNMODIFIED real text (windows of length L):\n")
print(f"{'L':>6} {'regime':>8} {'k':>5} {'#windows':>9} {'pass rate':>10} {'failures':>30}")
print("-" * 74)
for L in [50, 100, 200, 500, 1000, 2000]:
    for mode, klabel in [("const", "k=5"), ("grow", "k=L/8")]:
        total, passed, fails = sweep(corpus, L, mode)
        if total == 0: continue
        rate = passed / total
        k_shown = 5 if mode == "const" else max(2, (L + 7) // 8)
        failstr = ", ".join(f"{r}:{n}" for r, n in fails.items() if n) or "none"
        print(f"{L:>6} {klabel:>8} {k_shown:>5} {total:>9} {rate:>9.1%} {failstr:>30}")
print()
t2, p2, _ = sweep(corpus, 2000, "const")
print(f"Deployment case (L=2000, k=5, whistleblower scenario): {p2}/{t2} windows admissible = {p2/max(t2,1):.1%}")

Corpus: 9 articles, 103191 total tokens

Admissibility pass rate on UNMODIFIED real text (windows of length L):

     L   regime     k  #windows  pass rate                       failures
--------------------------------------------------------------------------
    50      k=5     5      2059    100.0%                           none
    50    k=L/8     7      2059    100.0%                           none
   100      k=5     5      1027    100.0%                           none
   100    k=L/8    13      1027    100.0%                           none
   200      k=5     5       512    100.0%                           none
   200    k=L/8    25       512    100.0%                           none
   500      k=5     5       201    100.0%                           none
   500    k=L/8    63       201    100.0%                           none
  1000      k=5     5        99    100.0%                           none
  1000    k=L/8   125        99    100.0%                           none
  2000  